In [ ]:
import warnings
import time
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Tuple, List, Dict, Optional
from pathlib import Path

import pandas as pd
import numpy as np
import optuna
import matplotlib.pyplot as plt
import seaborn as sns

import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import StepLR

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.style.use("seaborn-v0_8")
RANDOM_STATE: int = 42

| Item          | Version  |
|-------------------|---------|
| Kernel            | Python 3.12.0 |
| sklearn           | 1.7.1   |
| pandas            | 2.3.2   |
| seaborn           | 0.13.2  |
| matplotlib        | 3.10.6  |
| code formatter    | black 25.1.0  |

In [3]:
notebook_time = time.time()

#### 1. Download the  [Zindi gesture data](https://disk.360.yandex.ru/d/N_DIJPiWP2wBTw). Examine the data for some time. Make sure you get rid of duplicate examples.<br>Are there any mislabeled examples? Think about which extensions would be optimal for the current task.<br> Design a training and validation split. Use a random 33% of the total data as the validation set.

In [5]:
path: str = "data/African_Sign_Language_Gesture_Recognition_Challenge_Zindi/{name}.csv"
trn_df: pd.DataFrame = pd.read_csv(path.format(name="Train"))
tst_df: pd.DataFrame = pd.read_csv(path.format(name="Test"))

In [ ]:
# Duplicate img_IDs
trn_df.duplicated().sum()

np.int64(0)

#### 2. Write a custom pytorch dataset class for image and goal retrieval: read images using OpenCV2, extract targets using pandas.<br> Make sure the dataset class has the correct API for sample retrieval. Create a pytorch DataLoader for your datasets.

#### 3. Rebuild the LeNet-5 architecture. Use predefined layers from the pytorch library:<br> Conv2d, Pooling, Linear. You can use the Dropout layer to improve the quality of your model.

#### 4. Design a basic train-validation loop: iterate over the training dataset, batch by batch, update the parameters of the network,<br> and check the quality of the model using the validation set.<br> [Here](https://github.com/pytorch/examples/blob/main/mnist/main.py) you can find a comprehensive example of a basic training-validation pipeline (you can copy-paste it first, then modify it). <br>As loss for your model use Cross Entropy, as metric use ROC AUC score. Get ROC AUC higher than 0.75.

In [ ]:



class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)
        return output


def train(args, model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % args.log_interval == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))
            if args.dry_run:
                break


def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()  # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))


def main():
    # Training settings
    parser = argparse.ArgumentParser(description='PyTorch MNIST Example')
    parser.add_argument('--batch-size', type=int, default=64, metavar='N',
                        help='input batch size for training (default: 64)')
    parser.add_argument('--test-batch-size', type=int, default=1000, metavar='N',
                        help='input batch size for testing (default: 1000)')
    parser.add_argument('--epochs', type=int, default=14, metavar='N',
                        help='number of epochs to train (default: 14)')
    parser.add_argument('--lr', type=float, default=1.0, metavar='LR',
                        help='learning rate (default: 1.0)')
    parser.add_argument('--gamma', type=float, default=0.7, metavar='M',
                        help='Learning rate step gamma (default: 0.7)')
    parser.add_argument('--no-accel', action='store_true',
                        help='disables accelerator')
    parser.add_argument('--dry-run', action='store_true',
                        help='quickly check a single pass')
    parser.add_argument('--seed', type=int, default=1, metavar='S',
                        help='random seed (default: 1)')
    parser.add_argument('--log-interval', type=int, default=10, metavar='N',
                        help='how many batches to wait before logging training status')
    parser.add_argument('--save-model', action='store_true', 
                        help='For Saving the current Model')
    args = parser.parse_args()

    use_accel = not args.no_accel and torch.accelerator.is_available()

    torch.manual_seed(args.seed)

    if use_accel:
        device = torch.accelerator.current_accelerator()
    else:
        device = torch.device("cpu")

    train_kwargs = {'batch_size': args.batch_size}
    test_kwargs = {'batch_size': args.test_batch_size}
    if use_accel:
        accel_kwargs = {'num_workers': 1,
                        'persistent_workers': True,
                       'pin_memory': True,
                       'shuffle': True}
        train_kwargs.update(accel_kwargs)
        test_kwargs.update(accel_kwargs)

    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
        ])
    dataset1 = datasets.MNIST('../data', train=True, download=True,
                       transform=transform)
    dataset2 = datasets.MNIST('../data', train=False,
                       transform=transform)
    train_loader = torch.utils.data.DataLoader(dataset1,**train_kwargs)
    test_loader = torch.utils.data.DataLoader(dataset2, **test_kwargs)

    model = Net().to(device)
    optimizer = optim.Adadelta(model.parameters(), lr=args.lr)

    scheduler = StepLR(optimizer, step_size=1, gamma=args.gamma)
    for epoch in range(1, args.epochs + 1):
        train(args, model, device, train_loader, optimizer, epoch)
        test(model, device, test_loader)
        scheduler.step()

    if args.save_model:
        torch.save(model.state_dict(), "mnist_cnn.pt")


if __name__ == '__main__':
    main()

#### 5. Pick any vision model or backbone (resnet18 is recommended as a baseline) from this [library](https://github.com/rwightman/pytorch-image-models). <br>Change the head of your model to a linear layer with one output. Train the model for 2-4 epochs (iterations, traversals over the entire training dataset). <br>You must be able to obtain a ROC AUC greater than 0.9. You are also advised to play with other backbones to get better results.

#### 6. Apply different augmentations from the [albumentations](https://albumentations.ai/) library and check if they improve the validation score.

#### 7. Implement MixUp and CutMix augmentations and test them in your pipeline; check if they improve the validation score.

#### Bonus part.
#### B1. Add Test-Time-Augmentation (TTA) option to your code. <br>For example, try averaging the predictions of horizontally flipped and original images. What is your gain in terms of ROC AUC score?


#### * Sources
- adam https://arxiv.org/pdf/1412.6980

In [ ]:
total_seconds = time.time() - notebook_time
print(f"{total_seconds // 60}m {(total_seconds % 60):.1f}s")

15.0m 47.7s
